In [1]:
import os
import glob
from pathlib import Path
import cv2
import numpy as np
from ultralytics import YOLO
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
from datetime import datetime

print("Imports completed successfully!")
print("YOLO version:", YOLO.__version__ if hasattr(YOLO, '__version__') else "Unknown")

Imports completed successfully!
YOLO version: Unknown


In [ ]:
# Configuration
MODEL_PATH = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/saved_checkpoints/1_ryan_best.pt"
TEST_DATA_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/test_data"
OUTPUT_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result_2"

# Class names
CLASS_NAMES = ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']

# Create output directory (no subdirectories)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Created directory: {OUTPUT_DIR}")

print(f"Model path: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"Test data directory: {TEST_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print("Output format: TXT files with same name as images (YOLO format with confidence)")

Created directory: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result_2
Model path: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/saved_checkpoints/jimin_best_202507261159.pt
Model exists: True
Test data directory: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/test_data
Output directory: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result_2
Output format: TXT files with same name as images (YOLO format with confidence)


In [13]:
# Load the trained model
print("Loading YOLO model...")
model = YOLO(MODEL_PATH)
print(f"Model loaded successfully!")
print(f"Model classes: {model.names}")
print(f"Number of classes: {len(model.names)}")

Loading YOLO model...
Model loaded successfully!
Model classes: {0: 'Pothole', 1: 'Alligator Crack', 2: 'Transverse Crack', 3: 'Longitudinal Crack'}
Number of classes: 4


In [14]:
# Function to collect all test images
def get_all_test_images():
    """Collect all test images from all countries"""
    all_images = []
    
    for country in ['country_1', 'country_2', 'country_3']:
        country_path = os.path.join(TEST_DATA_DIR, country, 'images')
        if os.path.exists(country_path):
            image_files = glob.glob(os.path.join(country_path, '*.jpg'))
            for img_path in image_files:
                all_images.append({
                    'path': img_path,
                    'country': country,
                    'filename': os.path.basename(img_path)
                })
    
    return all_images

# Get all test images
test_images = get_all_test_images()
print(f"Total test images found: {len(test_images)}")

# Show distribution by country
for country in ['country_1', 'country_2', 'country_3']:
    country_count = len([img for img in test_images if img['country'] == country])
    print(f"{country}: {country_count} images")

Total test images found: 2961
country_1: 1024 images
country_2: 918 images
country_3: 1019 images


In [15]:
def run_inference_and_save(image_info, model, confidence_threshold=0.25):
    """
    Run inference on a single image and save the result as txt file in YOLO format with confidence
    Format: <class(0-3)> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>
    """
    image_path = image_info['path']
    filename = image_info['filename']
    
    # Get image dimensions
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error reading image: {image_path}")
        return None
    
    img_height, img_width = image.shape[:2]
    
    # Run inference
    results = model(image_path, conf=confidence_threshold, verbose=False)
    
    # Prepare output txt file path (same name as image but with .txt extension)
    txt_filename = os.path.splitext(filename)[0] + '.txt'
    output_txt_path = os.path.join(OUTPUT_DIR, txt_filename)
    
    detections = []
    
    # Open txt file for writing
    with open(output_txt_path, 'w') as f:
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            
            for box in boxes:
                # Get box coordinates in xyxy format
                xyxy = box.xyxy[0].cpu().numpy()
                confidence = box.conf[0].cpu().numpy()
                class_id = int(box.cls[0].cpu().numpy())
                
                # Convert xyxy to normalized center coordinates and dimensions
                x1, y1, x2, y2 = xyxy
                
                # Calculate center coordinates and dimensions
                x_center = (x1 + x2) / 2.0
                y_center = (y1 + y2) / 2.0
                width = x2 - x1
                height = y2 - y1
                
                # Normalize coordinates
                x_center_norm = x_center / img_width
                y_center_norm = y_center / img_height
                width_norm = width / img_width
                height_norm = height / img_height
                
                # Store detection info
                detections.append({
                    'class_id': class_id,
                    'class_name': model.names[class_id],
                    'confidence': float(confidence),
                    'x_center_norm': x_center_norm,
                    'y_center_norm': y_center_norm,
                    'width_norm': width_norm,
                    'height_norm': height_norm
                })
                
                # Write to txt file in the specified format
                f.write(f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f} {confidence:.6f}\n")
    
    return {
        'image_path': image_path,
        'output_path': output_txt_path,
        'detections': detections,
        'num_detections': len(detections)
    }

print("Inference function defined successfully!")
print("Output format: <class(0-3)> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>")

Inference function defined successfully!
Output format: <class(0-3)> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>


In [16]:
# Run inference on all test images
print("Starting inference on all test images...")
print(f"Processing {len(test_images)} images...")

inference_results = []
failed_images = []

# Set confidence threshold
CONFIDENCE_THRESHOLD = 0.10

# Process images with progress bar
for i, image_info in enumerate(tqdm(test_images, desc="Processing images")):
    try:
        result = run_inference_and_save(image_info, model, CONFIDENCE_THRESHOLD)
        if result is not None:
            inference_results.append(result)
        else:
            failed_images.append(image_info['path'])
    except Exception as e:
        print(f"Error processing {image_info['path']}: {str(e)}")
        failed_images.append(image_info['path'])

print(f"\nInference completed!")
print(f"Successfully processed: {len(inference_results)} images")
print(f"Failed to process: {len(failed_images)} images")

Starting inference on all test images...
Processing 2961 images...


Processing images: 100%|██████████| 2961/2961 [01:00<00:00, 49.07it/s]


Inference completed!
Successfully processed: 2961 images
Failed to process: 0 images
